# Previsão de Retorno para Revisão - Honda

**FGV - Modelagem Preditiva**

**Grupo:** João Victor Fortunato Teixeira Mundim, Rodrigo Pimentel Carneiro

## Objetivo
Prever se um cliente que comprou uma motocicleta Honda irá retornar para a primeira revisão programada.

## Modelos Implementados
1. Árvore de Decisão (Entropia/ID3)
2. Regressão Logística
3. Random Forest
4. Gradient Boosting

In [ ]:
###########################################
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
###########################################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_score, recall_score, f1_score)

import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas importadas com sucesso!")

In [ ]:
###########################################
# 2. CARREGAMENTO DOS DADOS
###########################################

# Para Google Colab, descomente:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/Honda_Revisao_Modelagem.csv')

# Para execução local:
df = pd.read_csv('../dados/Honda_Revisao_Modelagem.csv')

print("="*60)
print("DATASET: PREVISÃO DE RETORNO PARA REVISÃO - HONDA")
print("="*60)
print(f"\nRegistros: {df.shape[0]}")
print(f"Variáveis: {df.shape[1]}")
print(f"\nColunas: {list(df.columns)}")

df.head()

In [ ]:
###########################################
# 3. ANÁLISE EXPLORATÓRIA
###########################################
print("="*60)
print("ANÁLISE EXPLORATÓRIA")
print("="*60)

print("\n🎯 Variável Alvo - RETORNOU_REVISAO:")
print(df['RETORNOU_REVISAO'].value_counts())
print(f"\nTaxa de retorno: {df['RETORNOU_REVISAO'].mean()*100:.1f}%")

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribuição da variável alvo
colors = ['#e74c3c', '#2ecc71']
df['RETORNOU_REVISAO'].value_counts().plot(kind='bar', ax=axes[0,0], color=colors)
axes[0,0].set_title('Distribuição: Retornou para Revisão?')
axes[0,0].set_xticklabels(['Não Retornou (0)', 'Retornou (1)'], rotation=0)
axes[0,0].set_ylabel('Quantidade')

# 2. Taxa de retorno por família de moto
retorno_familia = df.groupby('MOTO_FAMILIA')['RETORNOU_REVISAO'].mean().sort_values(ascending=False)
retorno_familia.plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Taxa de Retorno por Família de Moto')
axes[0,1].set_ylabel('Taxa de Retorno')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Taxa de retorno por financiamento
retorno_fin = df.groupby('TEM_FINANCIAMENTO')['RETORNOU_REVISAO'].mean()
retorno_fin.plot(kind='bar', ax=axes[1,0], color=['#3498db', '#e67e22'])
axes[1,0].set_title('Taxa de Retorno: Com vs Sem Financiamento')
axes[1,0].set_xticklabels(['Sem Financiamento', 'Com Financiamento'], rotation=0)
axes[1,0].set_ylabel('Taxa de Retorno')

# 4. Taxa de retorno por cidade
retorno_cidade = df.groupby('ONDE_MOTO_VENDIDA')['RETORNOU_REVISAO'].mean().sort_values(ascending=False)
retorno_cidade.plot(kind='bar', ax=axes[1,1], color='teal')
axes[1,1].set_title('Taxa de Retorno por Cidade')
axes[1,1].set_ylabel('Taxa de Retorno')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
###########################################
# 4. ENGENHARIA DE VARIÁVEIS
###########################################
print("="*60)
print("ENGENHARIA DE VARIÁVEIS (FEATURE ENGINEERING)")
print("="*60)

# Separar features e target
X = df.drop(columns=['RETORNOU_REVISAO']).copy()
y = df['RETORNOU_REVISAO'].copy()

# Identificar colunas
colunas_cat = X.select_dtypes(include=['object']).columns.tolist()
colunas_num = X.select_dtypes(include=['number']).columns.tolist()

print(f"\nVariáveis Categóricas ({len(colunas_cat)}): {colunas_cat}")
print(f"\nVariáveis Numéricas ({len(colunas_num)}): {colunas_num}")

# Label Encoding para modelos de árvore
print("\n🔧 Aplicando Label Encoding...")
X_tree = X.copy()
encoders = {}
for col in colunas_cat:
    le = LabelEncoder()
    X_tree[col] = le.fit_transform(X_tree[col].astype(str))
    encoders[col] = le
    print(f"   {col}: {len(le.classes_)} classes")

# One-Hot Encoding + Scaling para Regressão Logística
print("\n🔧 Aplicando One-Hot Encoding para Regressão Logística...")
X_log = pd.get_dummies(X, columns=colunas_cat, drop_first=True)
scaler = StandardScaler()
X_log_scaled = pd.DataFrame(scaler.fit_transform(X_log), columns=X_log.columns)
print(f"   Features após One-Hot: {X_log_scaled.shape[1]}")

print("\n✅ Feature Engineering concluído!")

In [ ]:
###########################################
# 5. DIVISÃO TREINO/TESTE
###########################################
print("="*60)
print("DIVISÃO TREINO/TESTE (HOLDOUT)")
print("="*60)

random_state = 42
test_size = 0.2

# Para modelos de árvore
X_tree_train, X_tree_test, y_train, y_test = train_test_split(
    X_tree, y, test_size=test_size, random_state=random_state, stratify=y
)

# Para Regressão Logística
X_log_train, X_log_test, _, _ = train_test_split(
    X_log_scaled, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"\nDados de Treino: {len(y_train)} registros ({len(y_train)/len(y)*100:.0f}%)")
print(f"Dados de Teste:  {len(y_test)} registros ({len(y_test)/len(y)*100:.0f}%)")
print(f"\nDistribuição no Treino:")
print(f"   Não Retornou (0): {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")
print(f"   Retornou (1):     {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")

In [ ]:
###########################################
# 6. MODELO 1: ÁRVORE DE DECISÃO (ID3)
###########################################
print("="*60)
print("MODELO 1: ÁRVORE DE DECISÃO (ENTROPIA/ID3)")
print("="*60)

# Parâmetros testados
print("\n📋 Parâmetros testados:")
print("   criterion: entropy (ID3)")
print("   max_depth: [3, 5, 7, 10] → escolhido: 5")
print("   min_samples_split: [5, 10, 20] → escolhido: 20")
print("   min_samples_leaf: [5, 10, 20] → escolhido: 10")

# Modelo final
arvore = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=random_state
)

arvore.fit(X_tree_train, y_train)

# Avaliação
y_pred_arvore = arvore.predict(X_tree_test)
y_proba_arvore = arvore.predict_proba(X_tree_test)[:, 1]

print(f"\n📊 Resultados:")
print(f"   Acurácia Treino: {arvore.score(X_tree_train, y_train)*100:.2f}%")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_arvore):.4f}")
print(f"   Precisão:        {precision_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   Recall:          {recall_score(y_test, y_pred_arvore)*100:.2f}%")
print(f"   F1-Score:        {f1_score(y_test, y_pred_arvore)*100:.2f}%")

# Validação Cruzada
cv_arvore = cross_val_score(arvore, X_tree, y, cv=5, scoring='accuracy')
print(f"\n   Validação Cruzada (5-fold): {cv_arvore.mean()*100:.2f}% (+/- {cv_arvore.std()*100:.2f}%)")

In [ ]:
###########################################
# 7. MODELO 2: REGRESSÃO LOGÍSTICA
###########################################
print("="*60)
print("MODELO 2: REGRESSÃO LOGÍSTICA")
print("="*60)

print("\n📋 Parâmetros testados:")
print("   penalty: [l1, l2] → escolhido: l2")
print("   C: [0.1, 1.0, 10] → escolhido: 1.0")
print("   solver: lbfgs")
print("   max_iter: 1000")

logistica = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=random_state
)

logistica.fit(X_log_train, y_train)

y_pred_log = logistica.predict(X_log_test)
y_proba_log = logistica.predict_proba(X_log_test)[:, 1]

print(f"\n📊 Resultados:")
print(f"   Acurácia Treino: {logistica.score(X_log_train, y_train)*100:.2f}%")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_log)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_log):.4f}")
print(f"   Precisão:        {precision_score(y_test, y_pred_log)*100:.2f}%")
print(f"   Recall:          {recall_score(y_test, y_pred_log)*100:.2f}%")
print(f"   F1-Score:        {f1_score(y_test, y_pred_log)*100:.2f}%")

cv_log = cross_val_score(logistica, X_log_scaled, y, cv=5, scoring='accuracy')
print(f"\n   Validação Cruzada (5-fold): {cv_log.mean()*100:.2f}% (+/- {cv_log.std()*100:.2f}%)")

In [ ]:
###########################################
# 8. MODELO 3: RANDOM FOREST
###########################################
print("="*60)
print("MODELO 3: RANDOM FOREST")
print("="*60)

print("\n📋 Parâmetros testados:")
print("   n_estimators: [50, 100, 200] → escolhido: 100")
print("   criterion: [gini, entropy] → escolhido: entropy")
print("   max_depth: [5, 10, None] → escolhido: 10")
print("   max_features: sqrt")

rf = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    max_depth=10,
    max_features='sqrt',
    bootstrap=True,
    random_state=random_state,
    n_jobs=-1
)

rf.fit(X_tree_train, y_train)

y_pred_rf = rf.predict(X_tree_test)
y_proba_rf = rf.predict_proba(X_tree_test)[:, 1]

print(f"\n📊 Resultados:")
print(f"   Acurácia Treino: {rf.score(X_tree_train, y_train)*100:.2f}%")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"   Precisão:        {precision_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   Recall:          {recall_score(y_test, y_pred_rf)*100:.2f}%")
print(f"   F1-Score:        {f1_score(y_test, y_pred_rf)*100:.2f}%")

cv_rf = cross_val_score(rf, X_tree, y, cv=5, scoring='accuracy')
print(f"\n   Validação Cruzada (5-fold): {cv_rf.mean()*100:.2f}% (+/- {cv_rf.std()*100:.2f}%)")

In [ ]:
###########################################
# 9. MODELO 4: GRADIENT BOOSTING
###########################################
print("="*60)
print("MODELO 4: GRADIENT BOOSTING")
print("="*60)

print("\n📋 Parâmetros testados:")
print("   n_estimators: [50, 100, 150] → escolhido: 100")
print("   learning_rate: [0.05, 0.1, 0.2] → escolhido: 0.1")
print("   max_depth: [3, 5, 7] → escolhido: 3")
print("   subsample: 0.8")

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=random_state
)

gb.fit(X_tree_train, y_train)

y_pred_gb = gb.predict(X_tree_test)
y_proba_gb = gb.predict_proba(X_tree_test)[:, 1]

print(f"\n📊 Resultados:")
print(f"   Acurácia Treino: {gb.score(X_tree_train, y_train)*100:.2f}%")
print(f"   Acurácia Teste:  {accuracy_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   AUC-ROC:         {roc_auc_score(y_test, y_proba_gb):.4f}")
print(f"   Precisão:        {precision_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   Recall:          {recall_score(y_test, y_pred_gb)*100:.2f}%")
print(f"   F1-Score:        {f1_score(y_test, y_pred_gb)*100:.2f}%")

cv_gb = cross_val_score(gb, X_tree, y, cv=5, scoring='accuracy')
print(f"\n   Validação Cruzada (5-fold): {cv_gb.mean()*100:.2f}% (+/- {cv_gb.std()*100:.2f}%)")

In [ ]:
###########################################
# 10. COMPARAÇÃO DOS MODELOS
###########################################
print("="*60)
print("QUADRO COMPARATIVO DE DESEMPENHO")
print("="*60)

resultados = pd.DataFrame({
    'Modelo': ['Árvore de Decisão', 'Regressão Logística', 'Random Forest', 'Gradient Boosting'],
    'Acurácia Teste': [
        accuracy_score(y_test, y_pred_arvore),
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb)
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, y_proba_arvore),
        roc_auc_score(y_test, y_proba_log),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_gb)
    ],
    'Precisão': [
        precision_score(y_test, y_pred_arvore),
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_gb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_arvore),
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_gb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_arvore),
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_gb)
    ],
    'CV Média': [cv_arvore.mean(), cv_log.mean(), cv_rf.mean(), cv_gb.mean()],
    'CV Std': [cv_arvore.std(), cv_log.std(), cv_rf.std(), cv_gb.std()]
})

resultados = resultados.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print("\n")
display(resultados.style.format({
    'Acurácia Teste': '{:.2%}',
    'AUC-ROC': '{:.4f}',
    'Precisão': '{:.2%}',
    'Recall': '{:.2%}',
    'F1-Score': '{:.2%}',
    'CV Média': '{:.2%}',
    'CV Std': '{:.2%}'
}))

In [ ]:
###########################################
# 11. VISUALIZAÇÕES COMPARATIVAS
###########################################
print("="*60)
print("VISUALIZAÇÕES")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Curvas ROC
modelos_roc = [
    ('Árvore de Decisão', y_proba_arvore, 'blue'),
    ('Regressão Logística', y_proba_log, 'purple'),
    ('Random Forest', y_proba_rf, 'orange'),
    ('Gradient Boosting', y_proba_gb, 'green')
]

for nome, proba, cor in modelos_roc:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0,0].plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC={auc:.3f})')

axes[0,0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0,0].set_xlabel('Taxa de Falso Positivo')
axes[0,0].set_ylabel('Taxa de Verdadeiro Positivo')
axes[0,0].set_title('Curvas ROC Comparativas')
axes[0,0].legend(loc='lower right')
axes[0,0].grid(True, alpha=0.3)

# 2. Comparação de métricas
metricas = ['Acurácia Teste', 'Precisão', 'Recall', 'F1-Score']
x = np.arange(len(resultados))
width = 0.2

for i, metrica in enumerate(metricas):
    axes[0,1].bar(x + i*width, resultados[metrica], width, label=metrica)

axes[0,1].set_xticks(x + width*1.5)
axes[0,1].set_xticklabels(resultados['Modelo'], rotation=45, ha='right')
axes[0,1].set_ylabel('Valor')
axes[0,1].set_title('Comparação de Métricas por Modelo')
axes[0,1].legend()
axes[0,1].set_ylim(0, 1)

# 3. Matrizes de Confusão (melhor modelo)
melhor_idx = resultados['AUC-ROC'].idxmax()
melhor_modelo = resultados.loc[melhor_idx, 'Modelo']
melhor_pred = [y_pred_arvore, y_pred_log, y_pred_rf, y_pred_gb][melhor_idx]

cm = confusion_matrix(y_test, melhor_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,0],
            xticklabels=['Não Retornou', 'Retornou'],
            yticklabels=['Não Retornou', 'Retornou'])
axes[1,0].set_title(f'Matriz de Confusão - {melhor_modelo}')
axes[1,0].set_xlabel('Predito')
axes[1,0].set_ylabel('Real')

# 4. Importância das variáveis (Random Forest)
importancias = pd.DataFrame({
    'Feature': X_tree.columns,
    'Importância': rf.feature_importances_
}).sort_values('Importância', ascending=True).tail(10)

axes[1,1].barh(range(len(importancias)), importancias['Importância'], color='steelblue')
axes[1,1].set_yticks(range(len(importancias)))
axes[1,1].set_yticklabels(importancias['Feature'])
axes[1,1].set_xlabel('Importância')
axes[1,1].set_title('Top 10 Features Mais Importantes (Random Forest)')

plt.tight_layout()
plt.show()

In [ ]:
###########################################
# 12. CONCLUSÃO E ROTEIRO PARA PRODUÇÃO
###########################################
print("="*70)
print("CONCLUSÃO E ROTEIRO PARA PRODUÇÃO")
print("="*70)

melhor = resultados.iloc[0]

print(f"\n🏆 MELHOR MODELO: {melhor['Modelo']}")
print(f"\n📊 Desempenho:")
print(f"   • Acurácia: {melhor['Acurácia Teste']*100:.2f}%")
print(f"   • AUC-ROC: {melhor['AUC-ROC']:.4f}")
print(f"   • Precisão: {melhor['Precisão']*100:.2f}%")
print(f"   • Recall: {melhor['Recall']*100:.2f}%")
print(f"   • Validação Cruzada: {melhor['CV Média']*100:.2f}% (+/- {melhor['CV Std']*100:.2f}%)")

print(f"\n📋 ROTEIRO PARA PRODUÇÃO:")
print("""
1. ENTRADA DE DADOS:
   - Coletar dados do cliente no momento da venda
   - Variáveis necessárias: modelo da moto, família, cor, valor, 
     financiamento, cidade, acessórios (kit/voucher)

2. PRÉ-PROCESSAMENTO:
   - Aplicar Label Encoding nas variáveis categóricas
   - Usar os mesmos encoders do treinamento

3. PREDIÇÃO:
   - Executar o modelo treinado
   - Obter probabilidade de retorno para revisão

4. AÇÃO BASEADA NA PROBABILIDADE:
   ┌─────────────────────┬──────────────────────────────────────┐
   │ Probabilidade       │ Ação Recomendada                     │
   ├─────────────────────┼──────────────────────────────────────┤
   │ < 30%               │ ALTO RISCO: Contato proativo,        │
   │                     │ oferta de desconto na revisão        │
   ├─────────────────────┼──────────────────────────────────────┤
   │ 30% - 50%           │ MÉDIO RISCO: Lembrete reforçado      │
   │                     │ via WhatsApp/SMS                     │
   ├─────────────────────┼──────────────────────────────────────┤
   │ > 50%               │ BAIXO RISCO: Lembrete padrão         │
   └─────────────────────┴──────────────────────────────────────┘

5. DESEMPENHO ESPERADO:
   - O modelo identificará corretamente ~{:.0f}% dos clientes
   - Taxa de acerto nas ações de retenção: ~{:.0f}%
""".format(melhor['Acurácia Teste']*100, melhor['Precisão']*100))

print("="*70)